## ??:??????????
??notebook?,???????????????????????????????????,???????????????,???????????

## ??????????
1. ?????????
2. ???????
3. ???????
4. ??????,??????????
5. ???????,????????
6. ???????????
7. ?????????
8. ????,?????

???notebook??????1-3?,???????,?notebook???????4-6??

### ?????

In [ ]:
import pandas as pd
import numpy as np

# ????,????????????
import warnings
warnings.simplefilter('ignore')

pd.set_option('display.max_columns', 60)  # ????????60

# Matplotlib ???
import matplotlib.pyplot as plt
%matplotlib inline

plt.rcParams['font.size'] = 24  # ??????

from IPython.core.pylabtools import figsize  # ??????

import seaborn as sns  # ????
sns.set(font_scale=2)

# ?????????
from sklearn.preprocessing import Imputer, MinMaxScaler

# ??????
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# ?????
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

### ????
?????notebook???????

In [ ]:
# Read in data into dataframes 
train_features = pd.read_csv('data/training_features.csv')
test_features = pd.read_csv('data/testing_features.csv')
train_labels = pd.read_csv('data/training_labels.csv')
test_labels = pd.read_csv('data/testing_labels.csv')

# Display sizes of data
print('Training Feature Size: ', train_features.shape)
print('Testing Feature Size:  ', test_features.shape)
print('Training Labels Size:  ', train_labels.shape)
print('Testing Labels Size:   ', test_labels.shape)

In [ ]:
train_features.head()

### ?????

???????????????,?????XGBoost?LightGBM?????????,?XGBoost?????????????,??????????????????????????????????????????,?????????????????????????

???notebook???????50%????????,??????????????????????,?????????,????????????([????????????????](http://www.stat.columbia.edu/~gelman/arm/missing.pdf))?

??sklearn? Imputer object????????,??????????????????,????????????????????,?????????????,???????????,???[Data Leagage](https://www.kaggle.com/dansbecker/data-leakage)?

In [ ]:
# Create an imputer object with a median filling strategy
imputer = Imputer(strategy = 'median')

# Train on the training features
imputer.fit(train_features)

# Transform both training data and testing data
X = imputer.transform(train_features)
X_test = imputer.transform(test_features)

In [ ]:
print('Missing values in training features:', np.sum(np.isnan(X)))
print('Missing values in testing features:',np.sum(np.isnan(X_test)))

In [ ]:
# Make sure all values are finite
print(np.where(~np.isfinite(X)))
print(np.where(~np.isfinite(X_test)))

## ????

???????,????????????????????,???????????,?????????,????????(?????[?????????](https://github.com/ben1234560/AiLearning-Theory-Applying/blob/master/notebook_%E5%BF%85%E5%A4%87%E6%95%B0%E5%AD%A6%E5%9F%BA%E7%A1%80/%E5%9B%9E%E5%BD%92%E5%88%86%E6%9E%90%E7%AB%A0%E8%8A%82/%E6%A1%88%E4%BE%8B%EF%BC%9A%E6%B1%BD%E8%BD%A6%E4%BB%B7%E6%A0%BC%E9%A2%84%E6%B5%8B%E4%BB%BB%E5%8A%A1.ipynb))?

????????????????,?????,???????k???,????,??????????????????????,??????????,????????

????????:
* ?????,????????,??????????????????,?????????0,????1?
* ?????,????????,????????????????(??)?????????????0?1??,????????????

????????,??????????,?????????????????????,???????????????

In [ ]:
# Create the scaler object with a range of 0-1
scaler = MinMaxScaler(feature_range=(0, 1))

# Fit on the training data
scaler.fit(X)

# Transform both training data and testing data
X = scaler.transform(X)
X_test = scaler.transform(X_test)

In [ ]:
# Convert y to one-dimensional array (vector)
y = np.array(train_labels).reshape((-1, ))
y_test = np.array(test_labels).reshape((-1, ))

### ?????????(????)
    1. Linear Regression ????
    2. Support Vector Machine Regression ???????
    3. Random Forest Regression??????
    4. Gradient Boosting Regression GBDT??
    5. K-Nearest Neighbors Regression K????

?????????,?????

In [ ]:
# Function to calculation mean absolute error
def mae (y_true,y_pred):
    return np.mean(abs(y_true - y_pred))

# Takes in a model, trains the model, and evaluates the model on the test set
def fit_and_evaluate(model):
    
    # Train the model
    model.fit(X,y)
    
    # Make predictions and evalute
    model_pred = model.predict(X_test)
    model_mae = mae(y_test,model_pred)
    
    # Return the performance metric
    return model_mae

In [ ]:
lr = LinearRegression()
lr_mae = fit_and_evaluate(lr)

print('Linear Regression Performance on the test set: MAE = %0.4f'% lr_mae)

In [ ]:
svm = SVR(C=1000,gamma =0.1)
svm_mae = fit_and_evaluate(svm)

print('Support Vector Machine Regression Performance on the test set: MAE = %0.4f' % svm_mae)

In [ ]:
random_forest = RandomForestRegressor(random_state = 60)
random_forest_mae = fit_and_evaluate(random_forest)

print('Random Forest Regression Performance on the test set: MAE = %0.4f' % random_forest_mae)

In [ ]:
gradient_boosted = GradientBoostingRegressor(random_state=60)
gradient_boosted_mae = fit_and_evaluate(gradient_boosted)

print('Gradient Boosted Regression Performance on the test set: MAE = %0.4f' % gradient_boosted_mae)

In [ ]:
knn = KNeighborsRegressor(n_neighbors=10)
knn_mae = fit_and_evaluate(knn)

print('K-Nearest Neighbors Regression Performance on the test set: MAE = %0.4f' % knn_mae)

In [ ]:
plt.style.use('fivethirtyeight')
figsize(8, 4)

# Dataframe to hold the results
model_comparison = pd.DataFrame({'model':['Linear Regression',
                                          'Support Vector Machine',
                                          'Random Forest',
                                          'Gradient Boosted',
                                          'K-Nearest Neighbors'],
                                'mae':[lr_mae,
                                       svm_mae,
                                       random_forest_mae, 
                                       gradient_boosted_mae, 
                                       knn_mae]})
# Horizontal bar chart of test mae
model_comparison.sort_values('mae',ascending = False).plot(x = 'model',
                                                           y = 'mae',
                                                           kind = 'barh',
                                                           color = 'red', 
                                                           edgecolor = 'black')
# Plot formatting
plt.ylabel('');plt.yticks(size = 14);plt.xlabel('Mean Absolute Error');plt.xticks(size = 14)
plt.title('Model Comparison on Test MAE', size = 20);

?????????GBDT?????,????????????,?SVM?????????????????

??,?????,????????,????????,???????????????!

?????????XGBoost/LightGBM,?????????,XGBoost/LightGBM???GBDT,??GB??GBDT?????????,?????????XGBoost??LightGBM???,????GBDT??????????

?:????????????GBDT????????????,??????????

* 1.????:??????????(Bootstrap Sampling)???????????????,??????????????????????,GBDT???????,???????????????,??????

* 2.????:GBDT?????????,???????????????????????????????????????,??????????????

* 3.?????:?????????????????,????GBDT,?????????????GBDT?????????,????????,??????????????

In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBRegressor()
xgb_mae = fit_and_evaluate(xgb_model)

print('XGBoost Regression Performance on the test set: MAE = %0.4f' % xgb_mae)

In [ ]:
import lightgbm as lgb
lgb_model = lgb.LGBMRegressor()
lgb_mae = fit_and_evaluate(lgb_model)
print('lightgbm Regression Performance on the test set: MAE = %0.4f' % lgb_mae)

????XGBoost?LightGBM?????GBDT,?????????????????,?????????,?????GBDT?????????,????XGBoost?LightGBM,???????

## ???????????

**??????,??????????????????????????**

* ????????????????????,??????????????????????????,?K????????????
* ?????????????????,???????????

**[???????](http://scikit-learn.org/stable/modules/grid_search.html)????????????????**

* ????????????????????????,??????????????,????????????????????????????,?????????????(???)??????????????,?????????
* ??????????????????????????????????????????????,??????????????????????????????,????????????????????????????????????

### ????????????????

* ?????????????????????:???????,???????????????????????,????????????????????,?????????????????,???????,????????????????,????????????????????

* ??????????????????????K-Fold????,???????????????????,????????????????????????????K???,??????????,?????K-1???????,????K?????????????????K?,???????????????????????,???????????????????????????????K????????,???K?????????????????,???????????????,????????????????????

??k=5??k????????????:

<img src="data/kfold_cv.png" width="70%">

??????????????,??????????GBDT?????????????,??????????:?????????????,??4??????????,?????????????

In [ ]:
# ??????
# ????????
loss = ['ls','lad','huber']

# ??????????????
n_estimators = [100,500, 900, 1100,1500]

#  ??????
max_depth = [2,3,5,10,15]

# ??????????
min_samples_leaf = [1,2,4,6,8]

# ??????????
min_samples_split = [2, 4, 6, 10]

# ??????????????
max_features = ['auto', 'sqrt', 'log2', None]

# ?????????????
hyperparameter_grid = {'loss': loss,
                       'n_estimators': n_estimators,
                       'max_depth': max_depth,
                       'min_samples_leaf': min_samples_leaf,
                       'min_samples_split': min_samples_split,
                       'max_features': max_features}  

?????6??????????GBDT?????????????????,???????????,??????????????,???????????!??????,????[Scikit Learn??](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html#sklearn.ensemble.GradientBoostingRegressor)???,????????????????????,????????????????,?????????,??????????!

??????????,???????:

* `estimator`:?????
* `param_distributions`:?????????
* `cv`: ??k-fold????????
* `n_iter`:???????????
* `scoring`:???????
* `n_jobs`:????????(-1????????)
* `verbose`:??????(1??????)
* `return_train_score`:???????????????
* `random_state`:????????????,???????????

????????????????scikit?????????????,????????????????,??????????

#### ????

In [ ]:
# Create the model to use for hyperparameter tuning
model = GradientBoostingRegressor(random_state = 42)

# Set up the random search with 4-fold cross validation
random_cv = RandomizedSearchCV(estimator=model,
                               param_distributions=hyperparameter_grid,
                               cv=4, n_iter=25, 
                               scoring = 'neg_mean_absolute_error',
                               n_jobs = -1, verbose = 1, 
                               return_train_score = True,
                               random_state=42)

In [ ]:
# Fit on the training data
random_cv.fit(X, y)

In [ ]:
# Get all of the cv results and sort by the test performance
random_results = pd.DataFrame(random_cv.cv_results_).sort_values('mean_test_score', ascending = False)

random_results.head(10)

In [ ]:
random_cv.best_estimator_

??GBDT?????????:
* `loss = lad`
* `n_estimators = 500`
* `max_depth = 5`
* `min_samples_leaf = 6`
* `min_samples_split = 6`
* `max_features = None` 

????????????????????????,????????????,??????????

?????????????????????????,?????????????????????

???,????????????n_estimators(????),????????????,?????????????????????

### ????

????????????,??????????????

In [ ]:
# Create a range of trees to evaluate
trees_grid = {'n_estimators': [100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800]}

model = GradientBoostingRegressor(loss = 'lad', max_depth = 5,
                                  min_samples_leaf = 6,
                                  min_samples_split = 6,
                                  max_features = None,
                                  random_state = 42)

# Grid Search Object using the trees range and the random forest model
grid_search = GridSearchCV(estimator = model, param_grid=trees_grid, cv = 4, 
                           scoring = 'neg_mean_absolute_error', verbose = 1,
                           n_jobs = -1, return_train_score = True)

In [ ]:
# Fit the grid search
grid_search.fit(X, y)

In [ ]:
# Get the results into a dataframe
results = pd.DataFrame(grid_search.cv_results_)

# Plot the training and testing error vs number of trees
figsize(8, 8)
plt.style.use('fivethirtyeight')
plt.plot(results['param_n_estimators'], -1 * results['mean_test_score'], label = 'Testing Error')
plt.plot(results['param_n_estimators'], -1 * results['mean_train_score'], label = 'Training Error')
plt.xlabel('Number of Trees'); plt.ylabel('Mean Abosolute Error'); plt.legend();
plt.title('Performance vs Number of Trees');

In [ ]:
results.sort_values('mean_test_score', ascending = False).head(5)

?????,??????????!????????????,?????????????????,?????????????????????,???????????????????????????????,?????????????

??????????????????(?????????),??????????,???????????????????????????????????????????????GBDT????,??????????????????????????????????????????????GBDT????[???](http://blog.kaggle.com/2017/01/23/a-kaggle-master-explains-gradient-boosting/)???,????????????,???????????

### ?????????

??????????????????????????,????????????,??????????????,????????????,???????

??????,?????????????????

In [ ]:
# Default model
default_model = GradientBoostingRegressor(random_state = 42)

# Select the best model
final_model = grid_search.best_estimator_

final_model

In [ ]:
# ????,-n???????????,-r?????
%%timeit -n 1 -r 5
default_model.fit(X, y)

In [ ]:
%%timeit -n 1 -r 5
final_model.fit(X, y)

In [ ]:
default_pred = default_model.predict(X_test)
final_pred = final_model.predict(X_test)

print('Default model performance on the test set: MAE = %0.4f.' % mae(y_test, default_pred))
print('Final model performance on the test set:   MAE = %0.4f.' % mae(y_test, final_pred))

??????,??????,???????10%????????????????

In [ ]:
figsize(8, 8)

# Density plot of the final predictions and the test values
sns.kdeplot(final_pred, label = 'Predictions')
sns.kdeplot(y_test, label = 'Values')

# Label the plot
plt.xlabel('Energy Star Score'); plt.ylabel('Density');
plt.title('Test Values and Predictions');

????????????????

??????????????????,????????????,???????????(???)???????

In [ ]:
figsize = (6, 6)

# Calculate the residuals 
residuals = final_pred - y_test

# Plot the residuals in a histogram
plt.hist(residuals, color = 'red', bins = 20,
         edgecolor = 'black')
plt.xlabel('Error'); plt.ylabel('Count')
plt.title('Distribution of Residuals');

????????,????????????????????????????????

## ??
???????,?????????????????:

* ??????,??????????
* ???????,????????
* ???????????

????,??????????????,?????????????????????9.1?????????,??????????????,????????????????????????????,???????????????(??????)???????????????????????????,??????????????????????????

?????????????,????????????????????????????????????:????????????????????????,???????????????????????,????????????????????????,??????????????????????????????????,??????????????

????notebook?,???????????,????????????